In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — System dependencies + Python packages
# ════════════════════════════════════════════════════════════

!apt-get install -y ffmpeg -q

!pip install -q \
    flask==3.0.3 \
    flask-cors==4.0.1 \
    moviepy==1.0.3 \
    langchain==0.2.16 \
    langchain-openai==0.1.25 \
    langchain-community==0.2.16 \
    langchain-text-splitters==0.2.1 \
    chromadb==0.5.5 \
    sentence-transformers==3.0.1 \
    tensorflow==2.17.0 \
    keras==3.4.1 \
    pyngrok==7.2.0 \
    requests==2.32.3 \
    python-dotenv==1.0.1\
    gTTs

!pip install -q --upgrade tensorflow keras ml_dtypes

print("✅ Installation complete!")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Folder structure + .env + config.py
# ════════════════════════════════════════════════════════════

import os
from google.colab import userdata

# ── FOLDER STRUCTURE ─────────────────────────────────────────
os.makedirs("omnicontent/model", exist_ok=True)
os.makedirs("omnicontent/pipeline", exist_ok=True)
for path in ["omnicontent/__init__.py", "omnicontent/model/__init__.py", "omnicontent/pipeline/__init__.py"]:
    open(path, "w").close()

# ── .env (via Colab Secrets) ──────────────────────────────────
with open("omnicontent/.env", "w") as f:
    f.write(
        f"OPENROUTER_KEY={userdata.get('OPENROUTER_KEY')}\n"
        f"PEXELS_KEY={userdata.get('PEXELS_KEY')}\n"
        f"NGROK_TOKEN={userdata.get('NGROK_TOKEN')}\n"
        "PORT=5005\n"
    )

# ── config.py ──────────────────────────────────────────────
with open("omnicontent/config.py", "w") as f:
    f.write('''
import os
import logging
from dotenv import load_dotenv

load_dotenv(dotenv_path="omnicontent/.env")

OPENROUTER_KEY = os.environ.get("OPENROUTER_KEY", "")
PEXELS_KEY     = os.environ.get("PEXELS_KEY", "")
NGROK_TOKEN    = os.environ.get("NGROK_TOKEN", "")

PORT           = int(os.environ.get("PORT", 5005))

OUTPUT_VIDEO   = "omnicontent_output.mp4"
TEMP_AUDIO     = "temp_audio.mp3"
TEMP_VIDEO     = "temp_video.mp4"

VIRAL_THRESHOLD = 0.65
MODEL_PATH      = "omnicontent/model/viral_mlp.keras"

LLM_MODEL       = "openrouter/free"
LLM_BASE_URL    = "https://openrouter.ai/api/v1"
MAX_RETRIES     = 3

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s -> %(message)s",
    datefmt="%H:%M:%S",
)

def get_logger(name: str) -> logging.Logger:
    return logging.getLogger(name)
''')

print("✅ Folder structure, .env, and config.py created.")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — model/train.py + model/predictor.py
# ════════════════════════════════════════════════════════════

# ── model/train.py ─────────────────────────────────────────
with open("omnicontent/model/train.py", "w") as f:
    f.write('''
import numpy as np
import pandas as pd
import keras
from keras import layers
from sklearn.preprocessing import MinMaxScaler
import os, pickle

from omnicontent.config import MODEL_PATH, get_logger
log = get_logger("model.train")

DATA_PATH   = "/content/Social Media Engagement Dataset.csv"
SCALER_PATH = "omnicontent/model/scaler.pkl"


def load_and_prepare():
    df = pd.read_csv(DATA_PATH)

    df["hashtag_count"] = df["hashtags"].fillna("").apply(lambda x: len(str(x).split(",")))
    df["keyword_count"] = df["keywords"].fillna("").apply(lambda x: len(str(x).split(",")))
    df["text_length"]   = df["text_content"].fillna("").apply(len)
    df["has_emotion"]   = df["emotion_type"].notna().astype(int)

    features = [
        "sentiment_score",
        "toxicity_score",
        "hashtag_count",
        "keyword_count",
        "text_length",
        "has_emotion",
    ]

    df = df[features + ["engagement_rate"]].dropna()

    X = df[features].values
    y = df["engagement_rate"].values

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X = scaler_X.fit_transform(X)
    y = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

    os.makedirs(os.path.dirname(SCALER_PATH), exist_ok=True)
    with open(SCALER_PATH, "wb") as f:
        pickle.dump({"X": scaler_X, "y": scaler_y, "features": features}, f)

    log.info(f"Data prepared: {X.shape[0]} rows, {X.shape[1]} features")
    return X, y


def build_model(input_dim: int) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ], name="viral_mlp")
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


def train_and_save():
    X, y = load_and_prepare()
    log.info("Model training starting...")
    model = build_model(X.shape[1])
    history = model.fit(
        X, y,
        epochs=40,
        batch_size=64,
        validation_split=0.2,
        verbose=1,
    )
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    model.save(MODEL_PATH)
    log.info(f"Model saved -> {MODEL_PATH}")
    return model, history


if __name__ == "__main__":
    train_and_save()
''')

# ── model/predictor.py ──────────────────────────────────────
with open("omnicontent/model/predictor.py", "w") as f:
    f.write('''
import numpy as np
import pickle
import keras

from omnicontent.config import MODEL_PATH, get_logger
log = get_logger("model.predictor")

SCALER_PATH = "omnicontent/model/scaler.pkl"

_model = None
_scaler_data = None


def _load():
    global _model, _scaler_data
    if _model is None:
        _model = keras.models.load_model(MODEL_PATH)
        with open(SCALER_PATH, "rb") as f:
            _scaler_data = pickle.load(f)
        log.info("Model and scaler loaded.")
    return _model, _scaler_data


def predict_viral_score(keyword: str) -> float:
    model, scaler_data = _load()

    word_count = len(keyword.split())
    hashtag_count = max(1, word_count)
    keyword_count = word_count

    raw_features = np.array([[
        0.3,
        0.05,
        hashtag_count,
        keyword_count,
        len(keyword) * 8,
        1,
    ]])

    X_scaled = scaler_data["X"].transform(raw_features)
    y_scaled = model.predict(X_scaled, verbose=0)
    viral_score = scaler_data["y"].inverse_transform(y_scaled)[0][0]

    viral_score = float(np.clip(viral_score, 0, 1))
    log.info(f"Keyword='{keyword}' -> viral_score={viral_score:.3f}")
    return viral_score


def score_script_quality(script: dict) -> dict:
    issues = []
    score = 1.0

    voiceover = script.get("voiceover", "")
    scenes = script.get("scenes", [])

    if len(voiceover) > 500:
        issues.append("Voiceover exceeds 500 characters.")
        score -= 0.3

    if len(voiceover) < 50:
        issues.append("Voiceover is too short, insufficient content.")
        score -= 0.2

    if len(scenes) != 3:
        issues.append(f"Scene count should be 3, found {len(scenes)}.")
        score -= 0.3

    total_duration = sum(s.get("duration", 0) for s in scenes)
    if abs(total_duration - 30) > 5:
        issues.append(f"Total scene duration differs significantly from 30s: {total_duration}s")
        score -= 0.2

    score = max(0.0, score)
    return {"quality_score": round(score, 2), "issues": issues}
''')

print("✅ train.py and predictor.py created.")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — pipeline/agents.py
# ════════════════════════════════════════════════════════════

with open("omnicontent/pipeline/agents.py", "w") as f:
    f.write('''
import time
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
from langchain.chains import LLMChain

from omnicontent.config import OPENROUTER_KEY, LLM_MODEL, LLM_BASE_URL, get_logger

log = get_logger("pipeline.agents")

llm = ChatOpenAI(
    model=LLM_MODEL,
    openai_api_key=OPENROUTER_KEY,
    openai_api_base=LLM_BASE_URL,
)

log.info("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

_knowledge_base = [
    Document(page_content="Fitness content gets 40% more engagement when posted between 7-9 AM."),
    Document(page_content="Organic reach peaks on social media when using 3-5 relevant hashtags."),
    Document(page_content="The ideal video duration for TikTok and Reels is 15-30 seconds. Catch attention in the first 3 seconds."),
    Document(page_content="Using strong call-to-actions (CTA) like Comment below increases conversion rates by 25%."),
    Document(page_content="Food content performs best with close-up shots and satisfying sounds (ASMR-style)."),
    Document(page_content="Educational content with a clear hook in the first line retains 60% more viewers."),
    Document(page_content="Travel content benefits from drone shots and golden-hour lighting."),
    Document(page_content="Tech reviews perform best when they show the product in real use within the first 5 seconds."),
]

_chunks = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20).split_documents(_knowledge_base)
_vectorstore = Chroma.from_documents(_chunks, embeddings)
retriever = _vectorstore.as_retriever(search_kwargs={"k": 2})

log.info("RAG vector store ready.")

researcher_prompt = PromptTemplate(
    input_variables=["keyword", "context"],
    template=(
        "You are a social media researcher.\\n"
        "Extract viral content insights for the keyword: \\"{keyword}\\".\\n\\n"
        "Knowledge context:\\n{context}\\n\\n"
        "Provide 3-5 short, actionable bullet points."
    ),
)

scriptwriter_prompt = PromptTemplate(
    input_variables=["keyword", "research"],
    template=(
        "You are a viral TikTok scriptwriter.\\n"
        "Write a 30-second script for the keyword: \\"{keyword}\\".\\n\\n"
        "Research insights:\\n{research}\\n\\n"
        "Reply ONLY with this exact JSON:\\n"
        "{{\\n"
        "    \\"scenes\\": [\\n"
        "        {{\\"id\\": 1, \\"description\\": \\"visual scene description\\", \\"duration\\": 10}},\\n"
        "        {{\\"id\\": 2, \\"description\\": \\"visual scene description\\", \\"duration\\": 10}},\\n"
        "        {{\\"id\\": 3, \\"description\\": \\"visual scene description\\", \\"duration\\": 10}}\\n"
        "    ],\\n"
        "    \\"voiceover\\": \\"30-second narration text in English (max 500 characters)\\",\\n"
        "    \\"duration\\": 30\\n"
        "}}"
    ),
)

guard_prompt = PromptTemplate(
    input_variables=["script"],
    template=(
        "Review this video script:\\n{script}\\n\\n"
        "Criteria:\\n"
        "1. Is the voiceover under 500 characters?\\n"
        "2. Are there exactly 3 scenes?\\n\\n"
        "Return ONLY: {{\\"approved\\": true/false, \\"feedback\\": \\"error description if any\\"}}"
    ),
)

researcher_chain   = LLMChain(llm=llm, prompt=researcher_prompt)
scriptwriter_chain = LLMChain(llm=llm, prompt=scriptwriter_prompt)
guard_chain        = LLMChain(llm=llm, prompt=guard_prompt)


def _parse_json(text: str) -> dict:
    import re, json
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    match = re.search(r"\\{.*\\}", cleaned, re.DOTALL)
    if not match:
        raise ValueError(f"No valid JSON found in model output. Raw output: {text[:200]}")
    try:
        return json.loads(match.group())
    except Exception as e:
        raise ValueError(f"JSON parse error: {e}. Raw output: {text[:200]}")


def _invoke_with_retry(chain, inputs: dict, max_attempts: int = 4, base_delay: int = 15):
    for attempt in range(1, max_attempts + 1):
        try:
            response = chain.invoke(inputs)
            log.info(f"LLM response received (attempt {attempt}).")
            return response["text"]
        except Exception as e:
            is_rate_limit = "429" in str(e) or "rate" in str(e).lower()
            if is_rate_limit and attempt < max_attempts:
                wait = base_delay * attempt
                log.warning(f"Rate limit hit, waiting {wait}s (attempt {attempt}/{max_attempts})...")
                time.sleep(wait)
                continue
            raise


def run_agent_pipeline(keyword: str, max_retries: int = 3) -> dict:
    log.info(f"Starting RAG retrieval: {keyword!r}")
    context = "\\n".join(d.page_content for d in retriever.invoke(keyword))
    research = _invoke_with_retry(researcher_chain, {"keyword": keyword, "context": context})

    script = None
    for attempt in range(1, max_retries + 1):
        log.info(f"Script generation attempt {attempt}/{max_retries}...")
        raw = _invoke_with_retry(scriptwriter_chain, {"keyword": keyword, "research": research})
        try:
            script = _parse_json(raw)
        except ValueError as e:
            log.warning(f"Could not parse scriptwriter output: {e}")
            continue

        guard_raw = _invoke_with_retry(guard_chain, {"script": __import__("json").dumps(script)})
        try:
            guard = _parse_json(guard_raw)
        except ValueError:
            log.warning("Could not parse guard output, accepting script.")
            return script

        if guard.get("approved"):
            log.info("Script approved by Brand Guard.")
            return script

        log.info("Guard rejected: " + str(guard.get("feedback", "")))
        research += "\\nFEEDBACK: " + str(guard.get("feedback", ""))

    if script is None:
        raise RuntimeError(f"Could not generate a valid script in {max_retries} attempts.")

    log.warning("Guard approval not obtained, using last generated script.")
    return script
''')

print("✅ agents.py created.")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — pipeline/video.py
# ════════════════════════════════════════════════════════════

with open("omnicontent/pipeline/video.py", "w") as f:
    f.write('''
import requests
from moviepy.editor import VideoFileClip, AudioFileClip, vfx
from gtts import gTTS

from omnicontent.config import PEXELS_KEY, OUTPUT_VIDEO, TEMP_AUDIO, TEMP_VIDEO, get_logger

log = get_logger("pipeline.video")


def generate_voiceover(text: str) -> None:
    log.info("Generating voiceover (gTTS)...")
    tts = gTTS(text=text, lang="en")
    tts.save(TEMP_AUDIO)
    log.info(f"Voiceover saved -> {TEMP_AUDIO}")


def fetch_stock_video(query: str, max_attempts: int = 3) -> str:
    fallback_queries = [query, "lifestyle", "nature", "city"]
    for attempt, q in enumerate(fallback_queries[:max_attempts], start=1):
        log.info(f"Pexels query (attempt {attempt}/{max_attempts}): {q!r}")
        try:
            resp = requests.get(
                "https://api.pexels.com/videos/search",
                headers={"Authorization": PEXELS_KEY},
                params={"query": q, "per_page": 1, "orientation": "portrait"},
                timeout=15,
            )
            resp.raise_for_status()
            videos = resp.json().get("videos", [])
            if videos:
                video_url = videos[0]["video_files"][0]["link"]
                log.info(f"Video found: {q!r}")
                return video_url
            log.warning(f"No results found for '{q}', trying fallback.")
        except requests.RequestException as e:
            log.warning(f"Pexels request failed: {e}")
    raise RuntimeError("Could not find a video from Pexels in any query.")


def download_video(video_url: str) -> None:
    log.info("Downloading stock video...")
    resp = requests.get(video_url, timeout=60)
    resp.raise_for_status()
    with open(TEMP_VIDEO, "wb") as f:
        f.write(resp.content)
    log.info(f"Video downloaded -> {TEMP_VIDEO}")


def render_final_video() -> str:
    log.info("Rendering final video...")
    bg  = VideoFileClip(TEMP_VIDEO)
    sfx = AudioFileClip(TEMP_AUDIO)
    final = bg.fx(vfx.loop, duration=sfx.duration).resize((1080, 1920)).set_audio(sfx)
    final.write_videofile(OUTPUT_VIDEO, fps=24, logger=None)
    bg.close()
    sfx.close()
    final.close()
    log.info(f"Final video ready -> {OUTPUT_VIDEO}")
    return OUTPUT_VIDEO


def run_video_pipeline(script: dict, keyword: str = "") -> str:
    generate_voiceover(script["voiceover"])

    if keyword.strip():
        search_query = keyword.strip()
    else:
        first_scene = script["scenes"][0]["description"]
        search_query = " ".join(first_scene.split()[:3])

    video_url = fetch_stock_video(search_query)
    download_video(video_url)
    return render_final_video()
''')

print("✅ video.py created.")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Train the model
# ════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, ".")
from omnicontent.model.train import train_and_save

model, history = train_and_save()
print(f"\n✅ Model trained! Final val_mae: {history.history['val_mae'][-1]:.4f}")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — app.py: Flask routes + full UI (progress bar, viral
# score gauge, script preview, video player, footer credit)
# ════════════════════════════════════════════════════════════

app_content = """
import os
import threading
import traceback

from flask import Flask, request, jsonify, send_file
from flask_cors import CORS

from omnicontent.config import PORT, OUTPUT_VIDEO, get_logger
from omnicontent.model.predictor import predict_viral_score, score_script_quality
from omnicontent.pipeline.agents import run_agent_pipeline
from omnicontent.pipeline.video import run_video_pipeline

log = get_logger("app")

app = Flask(__name__)
CORS(app)


HTML_PAGE = \"\"\"<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>OmniContent AI</title>
<script src="https://cdn.tailwindcss.com"></script>
<style>
  body { font-family: 'Inter', system-ui, sans-serif; background: #0B0B12; }
  .gradient-text {
    background: linear-gradient(135deg, #A855F7, #EC4899);
    -webkit-background-clip: text;
    background-clip: text;
    color: transparent;
  }
  .gradient-bg {
    background: linear-gradient(135deg, #A855F7, #EC4899);
  }
  .glow {
    box-shadow: 0 0 40px rgba(168, 85, 247, 0.25);
  }
  .card {
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(255,255,255,0.08);
  }
  @keyframes pulse-bar {
    0%, 100% { opacity: 1; }
    50% { opacity: 0.6; }
  }
  .pulse { animation: pulse-bar 1.5s ease-in-out infinite; }
</style>
</head>
<body class="min-h-screen text-white flex items-center justify-center px-4 py-10">

  <div class="w-full max-w-md">

    <div class="text-center mb-8">
      <h1 class="text-3xl font-bold mb-1">
        <span class="gradient-text">OmniContent</span> AI
      </h1>
      <p class="text-gray-500 text-sm">Keyword in. Viral video out.</p>
    </div>

    <div class="card rounded-2xl p-6 glow">

      <label class="text-xs text-gray-400 uppercase tracking-wide mb-2 block">Keyword</label>
      <input id="keyword" type="text" placeholder="e.g. fitness motivation"
        class="w-full p-3 rounded-xl bg-black/30 border border-white/10 text-white mb-4 outline-none focus:border-purple-400 transition" />

      <button id="generateBtn" onclick="generate()"
        class="w-full gradient-bg p-3 rounded-xl font-semibold transition hover:opacity-90 active:scale-[0.99]">
        Generate Video
      </button>

      <div id="progress" class="mt-5 hidden">
        <div class="w-full bg-white/10 rounded-full h-1.5 overflow-hidden">
          <div id="bar" class="gradient-bg h-1.5 rounded-full transition-all duration-700 pulse" style="width:5%"></div>
        </div>
        <p id="step" class="text-xs text-gray-400 mt-2 text-center"></p>
      </div>

      <div id="errorBox" class="mt-4 hidden text-sm text-red-400 bg-red-500/10 border border-red-500/20 rounded-xl p-3"></div>

      <div id="result" class="mt-5 hidden">

        <div class="flex items-center justify-between mb-4">
          <span class="text-xs text-gray-400 uppercase tracking-wide">Viral Score</span>
          <span id="scoreValue" class="text-lg font-bold gradient-text">--</span>
        </div>
        <div class="w-full bg-white/10 rounded-full h-2 mb-5 overflow-hidden">
          <div id="scoreBar" class="gradient-bg h-2 rounded-full transition-all duration-1000" style="width:0%"></div>
        </div>

        <details class="mb-4 group">
          <summary class="text-xs text-gray-400 uppercase tracking-wide cursor-pointer select-none flex items-center justify-between">
            Script preview
            <span class="group-open:rotate-180 transition">&#9662;</span>
          </summary>
          <div id="scriptPreview" class="mt-3 space-y-2 text-sm text-gray-300"></div>
        </details>

        <video id="videoPlayer" class="w-full rounded-xl mb-4 hidden" controls></video>

        <a href="/download" download id="downloadBtn"
          class="block w-full text-center bg-white/10 hover:bg-white/20 border border-white/10 p-3 rounded-xl font-semibold transition">
          Download MP4
        </a>
      </div>

    </div>

    <p class="text-center text-xs text-gray-600 mt-6">
      Built by
      <a href="https://github.com/IhabAltekreeti" target="_blank"
         class="text-gray-400 hover:text-purple-400 transition underline underline-offset-2">
         Ihab Altekreeti
      </a>
    </p>

  </div>

<script>
const STEPS = [
  [8,  "Predicting viral score..."],
  [22, "Researching the topic..."],
  [40, "Writing the script..."],
  [58, "Brand guard review..."],
  [72, "Generating voiceover..."],
  [85, "Fetching footage..."],
  [95, "Rendering MP4..."],
];

async function generate() {
  const keyword = document.getElementById("keyword").value.trim();
  if (!keyword) return;

  const btn = document.getElementById("generateBtn");
  btn.disabled = true;
  btn.classList.add("opacity-50");

  document.getElementById("progress").classList.remove("hidden");
  document.getElementById("result").classList.add("hidden");
  document.getElementById("errorBox").classList.add("hidden");
  document.getElementById("videoPlayer").classList.add("hidden");

  let i = 0;
  const ticker = setInterval(() => {
    if (i < STEPS.length) {
      document.getElementById("bar").style.width = STEPS[i][0] + "%";
      document.getElementById("step").innerText = STEPS[i][1];
      i++;
    }
  }, 9000);

  try {
    const res = await fetch("/generate", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ keyword }),
    });
    const data = await res.json();
    clearInterval(ticker);

    if (data.success) {
      document.getElementById("bar").style.width = "100%";
      setTimeout(() => {
        document.getElementById("progress").classList.add("hidden");
        document.getElementById("result").classList.remove("hidden");

        const score = Math.round(data.viral_score * 100);
        document.getElementById("scoreValue").innerText = score + "%";
        setTimeout(() => {
          document.getElementById("scoreBar").style.width = score + "%";
        }, 100);

        const preview = document.getElementById("scriptPreview");
        preview.innerHTML = data.script.scenes.map((s, idx) =>
          `<div class="border-l-2 border-purple-500/40 pl-3"><span class="text-gray-500">Scene ${idx + 1}</span><br/>${s.description}</div>`
        ).join("");

        const video = document.getElementById("videoPlayer");
        video.src = "/download?t=" + Date.now();
        video.classList.remove("hidden");
      }, 400);
    } else {
      document.getElementById("progress").classList.add("hidden");
      const errBox = document.getElementById("errorBox");
      errBox.innerText = "Something went wrong: " + data.error;
      errBox.classList.remove("hidden");
    }
  } catch (err) {
    clearInterval(ticker);
    document.getElementById("progress").classList.add("hidden");
    const errBox = document.getElementById("errorBox");
    errBox.innerText = "Network error. Please try again.";
    errBox.classList.remove("hidden");
  } finally {
    btn.disabled = false;
    btn.classList.remove("opacity-50");
  }
}
</script>
</body>
</html>\"\"\"


@app.get("/")
def index():
    return HTML_PAGE


@app.get("/health")
def health():
    return jsonify({"status": "ok"})


@app.post("/generate")
def generate():
    data = request.json or {}
    keyword = data.get("keyword", "").strip()

    if not keyword:
        return jsonify({"success": False, "error": "Keyword cannot be empty."}), 400

    log.info(f"Generation started -> keyword={keyword!r}")
    try:
        viral_score = predict_viral_score(keyword)
        script = run_agent_pipeline(keyword)
        quality = score_script_quality(script)
        log.info(f"Script quality score: {quality}")

        run_video_pipeline(script, keyword=keyword)

        return jsonify({
            "success": True,
            "viral_score": viral_score,
            "script": script,
            "quality": quality,
        })
    except Exception as e:
        log.error(traceback.format_exc())
        return jsonify({"success": False, "error": str(e)}), 500


@app.get("/download")
def download():
    abs_path = os.path.abspath(OUTPUT_VIDEO)
    if not os.path.exists(abs_path):
        return jsonify({"error": "Video has not been generated yet."}), 404

    return send_file(
        abs_path,
        as_attachment=True,
        download_name="omnicontent_viral_video.mp4",
        mimetype="video/mp4"
    )

def run_app():
    app.run(host="0.0.0.0", port=PORT, use_reloader=False, threaded=True)
"""

with open("omnicontent/app.py", "w") as f:
    f.write(app_content)

print("✅ omnicontent/app.py created (Flask routes + UI)")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — Start the Flask app with an ngrok tunnel
# ════════════════════════════════════════════════════════════

import sys, threading, time
sys.path.insert(0, ".")

from pyngrok import ngrok
from omnicontent.config import NGROK_TOKEN, PORT
from omnicontent.app import run_app

# Clean up previous tunnels (if any)
ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(PORT)
print(f"\n🌍 App is ready -> {public_url}\n")

# Start Flask in the background (thread) so the Colab cell doesn't block
thread = threading.Thread(target=run_app, daemon=True)
thread.start()

time.sleep(2)
print("✅ Flask server is running. Click the link above.")